In [11]:
#Player # 1: Minimax(defensive)
#Player # 2: Expectimax(offensive)
#Player # 3: User/AI


import random
import numpy as np

class Card:
    def __init__(self, color, number):
        self.color = color
        self.number = number

    def is_skip(self):
        return self.number == "Skip"

    def __repr__(self):
        return f"{self.color} {self.number}"

def generate_deck():
    colors = ["Red", "Blue", "Green", "Yellow"]
    numbers = list(range(10)) + ["Skip"]

    deck = [Card(c, n) for c in colors for n in numbers]
    random.shuffle(deck)
    return deck

def deal_cards(deck):
    p1 = [deck.pop() for _ in range(5)]
    p2 = [deck.pop() for _ in range(5)]
    p3 = [deck.pop() for _ in range(5)]
    top = deck.pop()
    return p1, p2, p3, top, deck

In [12]:
def get_valid_moves(hand, top):
    if top is None:
        return []
    return [c for c in hand if c.color == top.color or c.number == top.number]

def apply_move(hand, move, deck, current_top):
    if move is None:
        if deck:
            hand.append(deck.pop())
        return current_top, False

    hand.remove(move)
    return move, move.is_skip()

class GameState:
    def __init__(self, p1, p2, p3, top, deck, turn):
        self.p1 = p1
        self.p2 = p2
        self.p3 = p3
        self.top = top
        self.deck = deck
        self.turn = turn

def clone(state):
    return GameState(
        state.p1.copy(),
        state.p2.copy(),
        state.p3.copy(),
        state.top,
        state.deck.copy(),
        state.turn
    )

In [13]:
def evaluate(state, player):
    if player == 0:
        hand = state.p1
        opp = (len(state.p2) + len(state.p3)) / 2
    elif player == 1:
        hand = state.p2
        opp = (len(state.p1) + len(state.p3)) / 2
    else:
        hand = state.p3
        opp = (len(state.p1) + len(state.p2)) / 2

    cai = len(hand)
    skip_cards = sum(1 for c in hand if c.is_skip())

    return 50 - 5*cai + 2*opp + 3*skip_cards

In [14]:
def minimax(state, depth, maximizing):
    if depth == 0 or len(state.p1) == 0:
        return evaluate(state, 0)

    if maximizing:
        best = -9999
        moves = get_valid_moves(state.p1, state.top)

        if not moves:
            new = clone(state)
            apply_move(new.p1, None, new.deck, new.top)
            return minimax(new, depth-1, False)

        for m in moves:
            new = clone(state)
            new.top, skip = apply_move(new.p1, m, new.deck, new.top)
            new.turn = 1 if not skip else 2
            best = max(best, minimax(new, depth-1, False))
        return best
    else:
        moves = get_valid_moves(state.p2, state.top)

        if not moves:
            new = clone(state)
            apply_move(new.p2, None, new.deck, new.top)
            return minimax(new, depth-1, True)

        m = random.choice(moves)
        new = clone(state)
        new.top, skip = apply_move(new.p2, m, new.deck, new.top)
        new.turn = 0 if not skip else 2
        return minimax(new, depth-1, True)


def expectimax(state, depth):
    if depth == 0 or len(state.p2) == 0:
        return evaluate(state, 1)

    if state.turn == 1:
        best = -9999
        moves = get_valid_moves(state.p2, state.top)

        if not moves:
            return expectimax_draw(state, depth)

        for m in moves:
            new = clone(state)
            new.top, skip = apply_move(new.p2, m, new.deck, new.top)
            new.turn = 2 if not skip else 0
            best = max(best, expectimax(new, depth-1))
        return best

    elif state.turn == 2:
        moves = get_valid_moves(state.p3, state.top)

        if not moves:
            new = clone(state)
            apply_move(new.p3, None, new.deck, new.top)
            new.turn = 1
            return expectimax(new, depth-1)

        m = random.choice(moves)
        new = clone(state)
        new.top, skip = apply_move(new.p3, m, new.deck, new.top)
        new.turn = 1 if not skip else 0
        return expectimax(new, depth-1)

    return expectimax(state, depth-1)


def expectimax_draw(state, depth):
    if not state.deck:
        return evaluate(state, 1)

    vals = []
    for card in state.deck:
        new = clone(state)
        new.p2.append(card)
        new.deck.remove(card)
        new.turn = 2
        vals.append(expectimax(new, depth-1))

    return np.mean(vals)

In [15]:
def user_move(hand, top):
    moves = get_valid_moves(hand, top)

    if not moves:
        print("No valid move → Drawing card")
        return None

    print("Your hand:")
    for i, c in enumerate(hand):
        print(i, ":", c)

    print("Valid moves:", moves)

    while True:
        choice = input("Enter index of card to play or 'd' to draw: ")
        if choice == 'd':
            return None
        if choice.isdigit():
            idx = int(choice)
            if 0 <= idx < len(hand) and hand[idx] in moves:
                return hand[idx]


def minimax_p3(state, depth, maximizing):
    if depth == 0 or len(state.p3) == 0:
        return evaluate(state, 2)

    if maximizing:
        best = -9999
        moves = get_valid_moves(state.p3, state.top)

        if not moves:
            new = clone(state)
            apply_move(new.p3, None, new.deck, new.top)
            return minimax_p3(new, depth-1, False)

        for m in moves:
            new = clone(state)
            new.top, skip = apply_move(new.p3, m, new.deck, new.top)
            new.turn = 0 if not skip else 1
            best = max(best, minimax_p3(new, depth-1, False))
        return best
    else:
        return evaluate(state, 2)

In [16]:
def print_game_tree(valid_moves):
    move_names = [str(m) for m in valid_moves]
    print("\n          AI (Root)")
    print("         /    |    \\")
    print(f"  {'   '.join(move_names)}")
    print("       /      |      \\")
    print("  Opponent  Opponent  Chance\n")


def play_game():
    mode = input("Select Mode: 1 = Manual, 2 = Simulation: ")

    deck = generate_deck()
    p1, p2, p3, top, deck = deal_cards(deck)
    state = GameState(p1, p2, p3, top, deck, 0)

    while True:
        print("\nTop:", state.top)

        if state.turn == 0:
            print("P1:", state.p1)
            moves = get_valid_moves(state.p1, state.top)

            if moves:
                print_game_tree(moves)
                best, best_val = None, -9999
                for m in moves:
                    new = clone(state)
                    new.top, _ = apply_move(new.p1, m, new.deck, new.top)
                    val = minimax(new, 2, False)
                    print("Move:", m, "Score:", val)
                    if val > best_val:
                        best, best_val = m, val

                state.top, skip = apply_move(state.p1, best, state.deck, state.top)
                state.turn = 1 if not skip else 2
            else:
                state.top, _ = apply_move(state.p1, None, state.deck, state.top)
                state.turn = 1

        elif state.turn == 1:
            print("P2:", state.p2)
            moves = get_valid_moves(state.p2, state.top)

            if moves:
                print_game_tree(moves)
                best, best_val = None, -9999
                for m in moves:
                    new = clone(state)
                    new.top, _ = apply_move(new.p2, m, new.deck, new.top)
                    val = expectimax(new, 2)
                    print("Move:", m, "Expected:", val)
                    if val > best_val:
                        best, best_val = m, val

                state.top, skip = apply_move(state.p2, best, state.deck, state.top)
                state.turn = 2 if not skip else 0
            else:
                state.top, _ = apply_move(state.p2, None, state.deck, state.top)
                state.turn = 2

        else:
            print("P3:", state.p3)

            if mode == "1":
                move = user_move(state.p3, state.top)
                state.top, skip = apply_move(state.p3, move, state.deck, state.top)
                state.turn = 0 if not skip else 1

            else:
                moves = get_valid_moves(state.p3, state.top)

                if moves:
                    print_game_tree(moves)
                    best, best_val = None, -9999
                    for m in moves:
                        new = clone(state)
                        new.top, _ = apply_move(new.p3, m, new.deck, new.top)
                        val = minimax_p3(new, 2, True)
                        print("Move:", m, "Score:", val)
                        if val > best_val:
                            best, best_val = m, val

                    state.top, skip = apply_move(state.p3, best, state.deck, state.top)
                    state.turn = 0 if not skip else 1
                else:
                    state.top, _ = apply_move(state.p3, None, state.deck, state.top)
                    state.turn = 0

        if len(state.p1) == 0:
            print("\nP1 Wins")
            break
        if len(state.p2) == 0:
            print("\nP2 Wins")
            break
        if len(state.p3) == 0:
            print("\nP3 Wins")
            break

play_game()

Select Mode: 1 = Manual, 2 = Simulation:  2



Top: Yellow 7
P1: [Green 4, Blue 7, Blue 5, Red 6, Green 6]

          AI (Root)
         /    |    \
  Blue 7
       /      |      \
  Opponent  Opponent  Chance

Move: Blue 7 Score: 44.0

Top: Blue 7
P2: [Yellow 5, Blue 6, Green 5, Red 0, Blue 0]

          AI (Root)
         /    |    \
  Blue 6   Blue 0
       /      |      \
  Opponent  Opponent  Chance

Move: Blue 6 Expected: 43.0
Move: Blue 0 Expected: 45.0

Top: Blue 0
P3: [Green 0, Red 7, Red 3, Green Skip, Red 1]

          AI (Root)
         /    |    \
  Green 0
       /      |      \
  Opponent  Opponent  Chance

Move: Green 0 Score: 43.0

Top: Green 0
P1: [Green 4, Blue 5, Red 6, Green 6]

          AI (Root)
         /    |    \
  Green 4   Green 6
       /      |      \
  Opponent  Opponent  Chance

Move: Green 4 Score: 47.0
Move: Green 6 Score: 47.0

Top: Green 4
P2: [Yellow 5, Blue 6, Green 5, Red 0]

          AI (Root)
         /    |    \
  Green 5
       /      |      \
  Opponent  Opponent  Chance

Move: Green 5